# The Sector Brain — a firing correlation network + a running record
Jake's idea: lay the sectors out like neurons in a circle, let each **fire** with its daily price move, wire
them by **correlation** (sectors that move together = connected), and **watch the money evolve** from Jan 2025.

**Metaphor → real method:** this is a *sector correlation network* (how systemic-risk research actually studies
markets). Nodes = 11 sectors. Node color = that day's return (green up / red down, intensity = size). Edges =
trailing correlation above a threshold (co-firing = wired). When the whole brain wires together → risk-on/off
regime; when it fragments into clusters firing against each other → rotation.

**Two halves — the second matters more:**
1. **The animation** = intuition candy (watch it evolve, generate hypotheses).
2. **The running record** = a dated table of the *quantified* state each day (returns, dispersion, breadth,
   avg correlation, leader/laggard, regime tag). **This is what lets you TEST a pattern instead of trusting
   your eye.** ⚠️ An animation *will* make you see patterns — most are apophenia (the data-mining trap). A
   pattern isn't real until it survives as a rule in the record.


In [ ]:
# 1 · Setup
!pip -q install yfinance >/dev/null 2>&1
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
import yfinance as yf
import matplotlib.pyplot as plt
from matplotlib import animation, cm, colors
from matplotlib.lines import Line2D
print("ready")


In [ ]:
# 2 · CONFIG
START      = "2025-01-01"   # 'watch the money evolve' from here
SECTORS = {"XLK":"Tech","XLC":"Comm","XLY":"Discr","XLF":"Fin","XLI":"Indu","XLE":"Enrgy",
           "XLB":"Matls","XLV":"Hlth","XLP":"Staples","XLU":"Utils","XLRE":"REIT"}
CENTER     = "SPY"          # center node = the market
CORR_WIN   = 30            # trailing days for the 'wiring' (correlation)
CORR_THRESH= 0.6           # draw an edge when trailing |corr| ≥ this
CLIP_RET   = 0.03          # color scale saturates at ±3% daily
MAKE_GIF   = True          # save the animation (gif). If it chokes, key-frame PNGs still save.
GIF_EVERY  = 1             # animate every Nth trading day (2-3 = smaller/faster gif)
print("CONFIG:", START, "|", len(SECTORS), "sectors | corr", CORR_WIN, "d ≥", CORR_THRESH)


In [ ]:
# 3 · Data + daily returns
tks = list(SECTORS)+[CENTER]
px = yf.download(tks, start=START, auto_adjust=True, progress=True)["Close"].dropna(how="all")
px = px.ffill().dropna()
ret = px.pct_change().dropna()
S = list(SECTORS)                       # sector order
print(f"{len(ret)} trading days, {ret.index[0].date()} → {ret.index[-1].date()}")


In [ ]:
# 4 · THE RUNNING RECORD — quantified state each day (this is the testable half)
rec = pd.DataFrame(index=ret.index)
for t in S: rec[t] = ret[t]
sec_ret = ret[S]
rec["breadth"]  = (sec_ret>0).mean(axis=1)                    # % sectors green
rec["disp"]     = sec_ret.std(axis=1)                         # cross-sectional dispersion (rotation gauge)
rec["mkt"]      = ret[CENTER]
rec["leader"]   = sec_ret.idxmax(axis=1)
rec["laggard"]  = sec_ret.idxmin(axis=1)
# rolling 'connectedness' = mean of upper-triangle trailing correlation, as of each day
avgc=[]
for i in range(len(ret)):
    if i<CORR_WIN: avgc.append(np.nan); continue
    c=sec_ret.iloc[i-CORR_WIN+1:i+1].corr().values
    iu=np.triu_indices_from(c,1); avgc.append(np.nanmean(c[iu]))
rec["avg_corr"]=avgc
# percentiles for regime tagging
rec["disp_pct"]=rec["disp"].rank(pct=True); rec["corr_pct"]=rec["avg_corr"].rank(pct=True)
def regime(r):
    if r.breadth>=0.8 and r.corr_pct>=0.5: return "BROAD risk-ON"
    if r.breadth<=0.2 and r.corr_pct>=0.5: return "BROAD risk-OFF / stress"
    if r.disp_pct>=0.75:                   return "ROTATION (high dispersion)"
    return "mixed / quiet"
rec["regime"]=rec.apply(regime,axis=1)
print(rec[["mkt","breadth","disp","avg_corr","leader","laggard","regime"]].tail(12).to_string())


In [ ]:
# 5 · PATTERNS in the record (test what the eye wants to claim)
print("="*66); print("REGIME MIX since",START); print("="*66)
print(rec["regime"].value_counts().to_string())
# leadership persistence: does today's leader lead tomorrow?
# NULL must be CONCENTRATION-ADJUSTED: leadership isn't uniform (XLE/XLK dominate), so consecutive-day
# repeats happen by concentration alone. Correct baseline = sum(p_i^2), NOT 1/N. (Fix: 2026-07-19.)
ld=rec["leader"]; persist=(ld.values[1:]==ld.values[:-1]).mean()
p=ld.value_counts(normalize=True); null=float((p**2).sum())     # concentration-adjusted independence null
se=(null*(1-null)/max(len(ld)-1,1))**0.5
excess=persist-null; sigma=excess/se if se>0 else float('nan')
print(f"\nLeadership persistence: today's leader leads tomorrow {persist:.1%} of days.")
print(f"  naive null 1/N = {1/len(S):.1%}  |  CONCENTRATION-ADJUSTED null Σp² = {null:.1%}  (the honest one)")
print(f"  excess vs adjusted null = {excess:+.1%}  ({sigma:+.1f}σ)  → "
      f"{'REAL momentum signal' if sigma>2 else 'NOISE — leadership does not persist beyond concentration'}")
print("Most-frequent daily leaders:"); print(ld.value_counts().head(5).to_string())
print("Most-frequent laggards:");      print(rec['laggard'].value_counts().head(5).to_string())
# notable days
print("\nTOP-5 DISPERSION days (biggest rotation):")
print(rec.nlargest(5,"disp")[["mkt","breadth","leader","laggard","regime"]].to_string())
print("\nTOP-5 CONNECTEDNESS days (brain most 'wired' = usually stress):")
print(rec.dropna(subset=["avg_corr"]).nlargest(5,"avg_corr")[["mkt","breadth","avg_corr","regime"]].to_string())


In [ ]:
# 6 · Layout + one-frame renderer (the brain)
ang=np.linspace(0,2*np.pi,len(S),endpoint=False)
POS={s:(np.cos(a),np.sin(a)) for s,a in zip(S,ang)}     # sectors on a circle
POS[CENTER]=(0.0,0.0)                                     # market in the center
cmap=cm.get_cmap("RdYlGn"); norm=colors.Normalize(-CLIP_RET,CLIP_RET)
def draw_day(ax,i):
    ax.clear(); ax.set_xlim(-1.35,1.35); ax.set_ylim(-1.35,1.35); ax.axis("off")
    d=ret.index[i]
    # edges = trailing correlation ≥ threshold
    if i>=CORR_WIN:
        c=sec_ret.iloc[i-CORR_WIN+1:i+1].corr()
        for a in range(len(S)):
            for b in range(a+1,len(S)):
                cc=c.iloc[a,b]
                if np.isfinite(cc) and cc>=CORR_THRESH:
                    x=[POS[S[a]][0],POS[S[b]][0]]; y=[POS[S[a]][1],POS[S[b]][1]]
                    ax.plot(x,y,color="#888",alpha=min(.6,(cc-CORR_THRESH)/(1-CORR_THRESH)*.6+.08),
                            lw=0.5+2*(cc-CORR_THRESH)/(1-CORR_THRESH),zorder=1)
    # nodes
    for s in S+[CENTER]:
        r=ret[s].iloc[i]; x,y=POS[s]
        sz=300 if s==CENTER else 260+4000*min(abs(r),CLIP_RET)
        ax.scatter([x],[y],s=sz,c=[cmap(norm(r))],edgecolor="k",lw=.6,zorder=3)
        lab=SECTORS.get(s,"MKT")
        ax.text(x*1.16,y*1.16,lab,ha="center",va="center",fontsize=8,zorder=4)
    ax.set_title(f"{d.date()}   mkt {ret[CENTER].iloc[i]:+.2%} | breadth {rec['breadth'].iloc[i]:.0%}"
                 f" | {rec['regime'].iloc[i]}",fontsize=11)
fig,ax=plt.subplots(figsize=(8,8)); draw_day(ax,len(ret)-1); plt.tight_layout(); plt.show()  # today


In [ ]:
# 7 · ANIMATE Jan 2025 → now (watch it evolve) + save
frames=list(range(0,len(ret),GIF_EVERY))
figA,axA=plt.subplots(figsize=(8,8))
def _f(k): draw_day(axA,frames[k]); return axA.collections
anim=animation.FuncAnimation(figA,_f,frames=len(frames),interval=120,blit=False)
saved=None
if MAKE_GIF:
    try:
        anim.save("sector_brain.gif",writer=animation.PillowWriter(fps=8)); saved="sector_brain.gif"
        print("saved",saved)
    except Exception as e:
        print("gif writer failed:",str(e)[:80])
# always drop 4 key frames (start / two mid / now) as PNGs
import numpy as _np
for k in _np.linspace(CORR_WIN,len(ret)-1,4).astype(int):
    figK,axK=plt.subplots(figsize=(7,7)); draw_day(axK,int(k))
    figK.savefig(f"brain_{ret.index[int(k)].date()}.png",dpi=90,bbox_inches="tight"); plt.close(figK)
print("saved 4 key-frame PNGs")
try:
    from IPython.display import Image, display
    if saved: display(Image(saved))
except Exception: pass


In [ ]:
# 8 · Export the running record
rec.to_csv("sector_brain_record.csv")
print("saved sector_brain_record.csv  (",len(rec),"days x",rec.shape[1],"cols )")
print("\nTHIS csv is the deliverable to mine — paste the regime mix + persistence + notable days back,")
print("and turn any eyeballed pattern into a falsifiable rule before trusting it.")


## How to read it — and how NOT to fool yourself
- **The animation** is for *intuition*: watch the brain wire up (edges thicken) into risk-off panics, fragment
  into clusters during rotation, and see which nodes lead. Generate hypotheses here — don't trade them here.
- **The record (`sector_brain_record.csv`)** is the truth-keeper. Every eyeballed pattern must become a
  testable claim against it: "when the brain fully wires (avg_corr top-decile), forward 5d SPY = ?"; "does the
  daily leader carry to tomorrow?" (cell 5 already answers that one). If it doesn't hold in the record, it was
  apophenia — the animation *manufactures* patterns, that's what moving pictures do to primate eyes.
- **Real signals this structure can carry** (test, don't assume): (1) **avg_corr spiking to extremes** →
  correlations-go-to-1 stress; (2) **dispersion regime** → rotation intensity ([[rotation-stickiness]]);
  (3) **leadership persistence** vs random → is there momentum in *which* sector leads.
- **Knobs:** `CORR_WIN` (wiring memory), `CORR_THRESH` (how many edges), `START`. Don't tune them until a
  claim tests green — same anti-data-mining rule as every other tool here.
- Next step if a pattern survives: add a **forward-return column** to the record and correlate the regime tag /
  avg_corr / dispersion with SPY's next 1/5/20d — that promotes "pretty" to "signal or not."
